# Parte 2 — Definição das categorias dos comentários

Este notebook implementa a definição das categorias temáticas dos comentários de NPS da Swift usando o **Método A — Clusterização / Descoberta Orientada por Dados**, conforme proposto no enunciado.

A técnica escolhida foi **LDA — Latent Dirichlet Allocation**, utilizada apenas como apoio exploratório para descoberta de tópicos. Nesta etapa **não é treinado nenhum classificador supervisionado**.

## Objetivo

Criar uma taxonomia inicial de categorias para os comentários, com base em padrões encontrados nos próprios textos.

A taxonomia será usada posteriormente para anotação manual e, só depois, para treinamento de um classificador supervisionado.

## Metodologia

Foi escolhido o **Método A** porque, neste momento, ainda não há duas anotações humanas independentes para validar uma taxonomia manual.

O LDA agrupa comentários por similaridade de palavras e retorna tópicos representados por termos mais frequentes. A interpretação final dos tópicos continua sendo humana: os tópicos são nomeados e, quando necessário, colapsados em categorias de negócio.

Foram testados modelos com diferentes números de tópicos:

- `K = 5`
- `K = 7`
- `K = 10`
- `K = 12`
- `K = 15`

A escolha final considerou três critérios:

1. **Perplexidade**: menor valor indica melhor ajuste estatístico;
2. **Coerência UMass**: valores mais próximos de zero indicam maior coerência entre palavras do tópico;
3. **Interpretabilidade de negócio**: os tópicos precisam fazer sentido como categorias acionáveis.

## 1. Configuração

In [1]:
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 180)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() in {'eda', 'notebooks', 'categorias'} else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'dataset_nps_preprocessamento_base.csv'
OUTPUT_DIR = PROJECT_ROOT / 'categorias'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Arquivo não encontrado: {DATA_PATH}')

DATA_PATH, OUTPUT_DIR

(WindowsPath('c:/Users/rafaelcruz-ieg/OneDrive - Instituto Germinare/Área de Trabalho/3 ANO/Ciencia de dados/nps-swift/nps-swift/data/processed/dataset_nps_preprocessamento_base.csv'),
 WindowsPath('c:/Users/rafaelcruz-ieg/OneDrive - Instituto Germinare/Área de Trabalho/3 ANO/Ciencia de dados/nps-swift/nps-swift/categorias'))

## 2. Leitura da base pré-processada

A entrada deste notebook é o arquivo gerado pela EDA. Assim, esta etapa não refaz a análise exploratória; apenas consome o dataset já preparado.

In [2]:
df = pd.read_csv(DATA_PATH)
df.shape

(128341, 19)

## 3. Seleção dos comentários para modelagem

Para evitar que o LDA seja distorcido por textos sem conteúdo temático, usamos flags criadas na EDA:

- `possivel_lixo`
- `possivel_nao_pt`
- `flag_curto`

Comentários muito curtos podem ser úteis para sentimento, mas geralmente não ajudam na descoberta de categorias temáticas.

In [3]:
text_col = 'comentario_limpo_basico'

base = df[df[text_col].fillna('').astype(str).str.strip().ne('')].copy()

for col in ['possivel_lixo', 'possivel_nao_pt', 'flag_curto']:
    if col in base.columns:
        base = base[~base[col].fillna(False).astype(bool)]

base = base.reset_index(drop=True)
base.shape

(96456, 19)

## 4. Pré-processamento textual

In [4]:
STOPWORDS = set('''
a o os as um uma uns umas de da do das dos em no na nos nas por para pra pro com sem sobre entre ate até e ou mas que se ao aos onde como quando porque pois
seu sua seus suas meu minha meus minhas nosso nossa nossos nossas esse essa esses essas este esta estes estas isso isto aquele aquela aqueles aquelas
foi foram era eram ser ter tem tinha tinham ha há ja já muito muita muitos muitas pouco pouca poucos poucas mais menos bem mal
bom boa bons boas ruim ruins otimo ótima ótimo otimos ótimos pessimo péssima péssimo pessimos péssimos
loja lojas produto produtos comprar compra cliente clientes swift mercado mercados carne carnes atendimento atendente dia dias vez vezes tudo nada sempre
vcs vc voce você voces vocês gente mim aqui ali la lá ainda tambem também agora hoje ontem
'''.split())

for negacao in ['nao', 'não', 'nunca', 'jamais']:
    STOPWORDS.discard(negacao)


def normalizar_texto(texto):
    texto = str(texto).lower().strip()
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')
    texto = re.sub(r'http\S+|www\.\S+', ' ', texto)
    texto = re.sub(r'[^a-z0-9\s]', ' ', texto)
    return re.sub(r'\s+', ' ', texto).strip()

base['texto_lda'] = base[text_col].map(normalizar_texto)
base = base[base['texto_lda'].ne('')].reset_index(drop=True)
base[['comentario_limpo_basico', 'texto_lda']].head()

,comentario_limpo_basico,texto_lda
0,Bastante falta. De produto. Talvez pela data. Mas não tinha muita coisa. Principalmente bovino,bastante falta de produto talvez pela data mas nao tinha muita coisa principalmente bovino
1,"Estrutura excelente, mas, os peços deveriam ser melhores ou equivalentes aos mercados de carnes",estrutura excelente mas os pecos deveriam ser melhores ou equivalentes aos mercados de carnes
2,Alguns produtos com porções menores.,alguns produtos com porcoes menores
3,Os preços estão subindo muito rapidamente e o peso dos produtos nas embalagens estão diminuindo.,os precos estao subindo muito rapidamente e o peso dos produtos nas embalagens estao diminuindo
4,Seria muito interessante ter a opção de adquirir a carne fatiada na hora.,seria muito interessante ter a opcao de adquirir a carne fatiada na hora


## 5. Vetorização

O LDA foi aplicado sobre uma representação Bag of Words com unigramas e bigramas. Isso permite capturar tanto palavras isoladas quanto expressões como “preço alto”, “falta produto” ou “entrega atrasada”.

In [5]:
vectorizer = CountVectorizer(
    stop_words=list(STOPWORDS),
    min_df=10,
    max_df=0.70,
    max_features=5000,
    ngram_range=(1, 2),
    token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z]{2,}\b'
)

X = vectorizer.fit_transform(base['texto_lda'])
terms = np.array(vectorizer.get_feature_names_out())
X.shape

(96456, 5000)

## 6. Teste de quantidade de tópicos

In [6]:
def palavras_topico(model, top_n=12):
    return [terms[topic.argsort()[::-1][:top_n]].tolist() for topic in model.components_]


def coerencia_umass(topic_words, X_binary):
    term_to_idx = {term: i for i, term in enumerate(terms)}
    doc_freq = np.asarray(X_binary.sum(axis=0)).ravel() + 1
    scores = []

    for words in topic_words:
        idxs = [term_to_idx[w] for w in words]
        pares = []
        for i in range(1, len(idxs)):
            for j in range(i):
                wi, wj = idxs[i], idxs[j]
                cooc = X_binary[:, wi].multiply(X_binary[:, wj]).sum() + 1
                pares.append(np.log(cooc / doc_freq[wj]))
        scores.append(np.mean(pares))

    return float(np.mean(scores))


X_binary = X.copy()
X_binary.data = np.ones_like(X_binary.data)

K_VALUES = [5, 7, 10, 12, 15]
models = {}
metrics = []

for k in K_VALUES:
    lda = LatentDirichletAllocation(
        n_components=k,
        random_state=42,
        learning_method='batch',
        max_iter=20,
        n_jobs=-1,
    )
    lda.fit(X)
    topic_words = palavras_topico(lda)
    models[k] = lda
    metrics.append({
        'k_topicos': k,
        'perplexidade': lda.perplexity(X),
        'coerencia_umass': coerencia_umass(topic_words, X_binary),
    })

metricas = pd.DataFrame(metrics)
metricas

KeyboardInterrupt: 

## 7. Escolha do modelo final

Apesar de modelos com mais tópicos apresentarem perplexidade menor, `K=7` foi escolhido porque gerou tópicos mais interpretáveis e próximos das categorias de negócio esperadas.

In [ ]:
K_FINAL = 7
lda_final = models[K_FINAL]

## 8. Palavras principais por tópico

In [ ]:
palavras_rows = []
for topico_id, topic in enumerate(lda_final.components_):
    for rank, idx in enumerate(topic.argsort()[::-1][:20], 1):
        palavras_rows.append({
            'topico_id': topico_id,
            'rank': rank,
            'termo': terms[idx],
            'peso': topic[idx],
        })

topicos_palavras = pd.DataFrame(palavras_rows)
topicos_palavras.head(20)

,topico_id,rank,termo,peso
0,0,1,nao,7786.435799
1,0,2,entrega,5093.985149
2,0,3,pedido,4407.542332
3,0,4,tive,1885.802131
4,0,5,valor,1427.202793
5,0,6,recebi,1362.142002
6,0,7,casa,1199.894928
7,0,8,entregue,1162.142502
8,0,9,problema,1072.427953
9,0,10,veio,1070.693046


## 9. Exemplos reais por tópico

In [ ]:
topic_probs = lda_final.transform(X)
base['topico_id'] = topic_probs.argmax(axis=1)
base['prob_topico'] = topic_probs.max(axis=1)

exemplos_rows = []
for topico_id in sorted(base['topico_id'].unique()):
    exemplos = base[base['topico_id'].eq(topico_id)].sort_values('prob_topico', ascending=False).head(12)
    for _, row in exemplos.iterrows():
        exemplos_rows.append({
            'topico_id': topico_id,
            'prob_topico': row['prob_topico'],
            'classificacao': row['classificacao'],
            'comentario': row[text_col],
        })

exemplos_por_topico = pd.DataFrame(exemplos_rows)
exemplos_por_topico.to_csv(OUTPUT_DIR / 'exemplos_por_topico_lda.csv', index=False)
exemplos_por_topico.head(20)

,topico_id,prob_topico,classificacao,comentario
0,0,0.989259,detrator,"Minha experiência foi péssima! Agendei a entrega para o período da manhã, recebi uma mensagem informando que estava a caminho, depois que foi entregue e não havia sido entregue..."
1,0,0.988694,detrator,"Ja solicitamos 3x pedido on-line. No primeiro, deu tudo certo, chegou no primeiro horario conforme o agendamento e tudo certinho. No segundo, tivemos que alterar a data e entre..."
2,0,0.985917,detrator,"Fiz a compra para que fosse entregue domingo 12/01, até 12h. Ocorre que não recebi meu pedido conforme o frete pago, a central de relacionamento não atende aos domingos. Na seg..."
3,0,0.984673,neutro,"Tive um probleminha com a entrega. Primeiro disseram que o pedido estava a caminho. Aguardei. Aí recebi uma mensagem dizendo que eu não estava em casa para receber o pedido, o ..."
4,0,0.984292,detrator,Primeiro não cumpriram o prazo. Depois falaram que o entregador passou aqui não tinha ninguém sendo que ele nem tocou a campainha. Depois falaram que eu atrasar a entrega e eu ...
5,0,0.984040,neutro,"Fiz uma compra com entrega em até 5h00 e paguei o frete correspondente, a compra não foi entregue nem no mesmo dia apenas no dia seguinte consegui contato e me informaram que h..."
6,0,0.983494,neutro,"Atenção para realizar o prazo de entrega. Realizei a compra no domingo 2/5/25 para entregar no primeiro dia útil (segunda-feira), mas entregue somente no segundo dia útil (terç..."
7,0,0.983177,promotor,Quanto ao prazo tive problemas. Fiz o pedido as 11h50 teria que ter sido entregue no máximo as 16h50. Entrei em contato com a Swift Resolve em torno de 17h30 para saber do meu ...
8,0,0.983136,detrator,"Sinceramente, em tudo! Recebi meu pedido incompleto, não recebi a nota fiscal para conferir peso e a quantidade. A entrega agendada atrasou 1 dia. Em resumo, não sei se estou s..."
9,0,0.982836,detrator,recentimente fiz um pedido no dia 26/07 para entregar no mesmo dia. Não foi entregue e a Swift não deu satisfação por que não entregou o pedido. Na segunda feira dia 28 entrara...


## 10. Interpretação dos tópicos

A interpretação final dos tópicos foi feita a partir das palavras principais e dos exemplos reais.

Decisões:

- tópicos `0` e `2` foram colapsados em **Atendimento / Loja física**;
- tópicos `1` e `5` foram colapsados em **Preço / Promoções**;
- tópico `3` foi mantido como **Qualidade do produto**;
- tópico `4` foi mantido como **App / Site / Pagamento**;
- tópico `6` foi mantido como **Entrega / Pedido digital**.

Nenhum tópico foi descartado integralmente como ruído. Comentários genéricos ou sem tema claro serão tratados como **Outros / Genéricos** na anotação manual.

In [ ]:
mapa_topico_categoria = {
    0: 'Atendimento / Loja física',
    1: 'Preço / Promoções',
    2: 'Atendimento / Loja física',
    3: 'Qualidade do produto',
    4: 'App / Site / Pagamento',
    5: 'Preço / Promoções',
    6: 'Entrega / Pedido digital',
}

criterios_categoria = {
    'Atendimento / Loja física': 'Comentários sobre atendimento, funcionários, organização da loja, caixa, fila e experiência presencial.',
    'Preço / Promoções': 'Comentários sobre preço, promoções, descontos, valor percebido, custo-benefício e comparação com mercado.',
    'Qualidade do produto': 'Comentários sobre qualidade, sabor, textura, corte, gordura, validade e conservação dos produtos.',
    'App / Site / Pagamento': 'Comentários sobre app, site, pagamento, estoque online, jornada digital e problemas no checkout.',
    'Entrega / Pedido digital': 'Comentários sobre entrega, pedido, retirada, produto não recebido, atraso, estorno e problemas operacionais.',
    'Outros / Genéricos': 'Comentários sem tema acionável ou que não se encaixem nas categorias anteriores.',
}

base['categoria_final'] = base['topico_id'].map(mapa_topico_categoria)

## 11. Taxonomia final

In [ ]:
taxonomia_rows = []
for categoria, grupo in base.groupby('categoria_final'):
    exemplos_categoria = grupo.sort_values('prob_topico', ascending=False).head(3)[text_col].tolist()
    topicos_origem = sorted(grupo['topico_id'].unique().tolist())
    taxonomia_rows.append({
        'categoria': categoria,
        'topicos_origem': ', '.join(map(str, topicos_origem)),
        'descricao_criterio_inclusao': criterios_categoria[categoria],
        'exemplos': ' | '.join(exemplos_categoria),
        'qtd_comentarios': len(grupo),
        'percentual_estimado': round(len(grupo) / len(base) * 100, 2),
    })

taxonomia_rows.append({
    'categoria': 'Outros / Genéricos',
    'topicos_origem': 'fallback na anotação manual',
    'descricao_criterio_inclusao': criterios_categoria['Outros / Genéricos'],
    'exemplos': 'está tudo perfeito | sem comentários | nota 10',
    'qtd_comentarios': 0,
    'percentual_estimado': 0.0,
})

taxonomia = pd.DataFrame(taxonomia_rows).sort_values('percentual_estimado', ascending=False)
taxonomia.to_csv(OUTPUT_DIR / 'taxonomia_metodo_a_lda.csv', index=False)
taxonomia

,categoria,topicos_origem,descricao_criterio_inclusao,exemplos,qtd_comentarios,percentual_estimado
3,Preço / Promoções,"1, 5","Comentários sobre preço, promoções, descontos, valor percebido, custo-benefício e comparação com mercado.","Poderiam criar um programa de fidelidade ou pontuação para os clientes, seja da loja ou de alguns produtos, pois assim compensaria a questão de muitos produtos que estão com pr...",27723,28.74
1,Atendimento / Loja física,"0, 2","Comentários sobre atendimento, funcionários, organização da loja, caixa, fila e experiência presencial.","Minha experiência foi péssima! Agendei a entrega para o período da manhã, recebi uma mensagem informando que estava a caminho, depois que foi entregue e não havia sido entregue...",24432,25.33
4,Qualidade do produto,3,"Comentários sobre qualidade, sabor, textura, corte, gordura, validade e conservação dos produtos.","Sempre sou bem atendido, produtos sempre de primeira tanto em qualidade quanto em preço recomendo sempre para amigos e familiares, em especial a unidade do vila branca sempre f...",21178,21.96
0,App / Site / Pagamento,4,"Comentários sobre app, site, pagamento, estoque online, jornada digital e problemas no checkout.","Não recomendo para ninguém a loja do bairro de Veleiros em SP Atendimento ruim, caixas sem preparação, confusos, desorganizados, sem respeito com o cliente. Nunca faço avaliaçõ...",12959,13.44
2,Entrega / Pedido digital,6,"Comentários sobre entrega, pedido, retirada, produto não recebido, atraso, estorno e problemas operacionais.","Comprei pera do colchão mole e veio com muito, muito mesmo, de nervos e sebos, coisas que nunca vi em carne deste tipo.... Comprei picanha (não era da linha combo) e veio com m...",10164,10.54
5,Outros / Genéricos,fallback na anotação manual,Comentários sem tema acionável ou que não se encaixem nas categorias anteriores.,está tudo perfeito | sem comentários | nota 10,0,0.00


## 12. Decisões obrigatórias

**Tipo de classificação:** single-label.

Cada comentário será associado a uma única categoria, correspondente ao tema dominante ou mais acionável para negócio. Essa escolha simplifica a anotação, reduz ambiguidade e facilita o treinamento supervisionado posterior.

**Comentários fora de categoria:** serão classificados como **Outros / Genéricos**, não descartados automaticamente.

**Categorias com baixo volume:** caso a anotação manual revele alguma categoria com volume muito baixo, ela deve ser fundida com a categoria semanticamente mais próxima.

**Distribuição:** a distribuição estimada pelo LDA não ficou perfeitamente equilibrada, mas nenhuma categoria principal ficou com volume inviável para continuidade da taxonomia.

## 13. Amostra para anotação manual posterior

O documento pede amostra anotada e Kappa de Cohen. Como o Kappa exige dois anotadores independentes, esta etapa prepara apenas a amostra de 200 comentários para anotação posterior.

In [ ]:
amostras = []
comentarios_validos = df[df[text_col].fillna('').astype(str).str.strip().ne('')]

for classe, grupo in comentarios_validos.groupby('classificacao'):
    n = max(1, round(200 * len(grupo) / len(comentarios_validos)))
    amostras.append(grupo.sample(n=min(n, len(grupo)), random_state=42))

amostra_anotacao = pd.concat(amostras).sample(frac=1, random_state=42).head(200).reset_index(drop=True)
amostra_anotacao.insert(0, 'id_comentario', range(1, len(amostra_anotacao) + 1))
amostra_anotacao = amostra_anotacao[['id_comentario', 'classificacao', text_col]].rename(columns={text_col: 'comentario'})
amostra_anotacao['categoria_anotador_1'] = ''
amostra_anotacao['categoria_anotador_2'] = ''
amostra_anotacao.to_csv(OUTPUT_DIR / 'amostra_para_anotacao_categorias.csv', index=False)
amostra_anotacao.head()

,id_comentario,classificacao,comentario,categoria_anotador_1,categoria_anotador_2
0,1,promotor,Tornar os combos e promoções mais frequentes e acessíveis; no atendimento obter mais divulgação das promoções e novidades,,
1,2,detrator,"muita fila na loja do Portal, tem um monte de funcionário, mas só deixa um no caixa. O preço da carne aumentou muito, não está mais tendo diferença da fresca para a congelada, ...",,
2,3,neutro,"mais carnes diferentes, avestruz, rã, javali, tipo assim",,
3,4,promotor,Poderia aceitar vale alimentação como forma de pagamento no site e no aplicativo.,,
4,5,promotor,O produto é bom. As funcionárias atendem com muita simpatia e conhecimento,,


## 14. Categorias válidas para anotação

In [ ]:
categorias_validas = pd.DataFrame({
    'categoria': [
        'Atendimento / Loja física',
        'Preço / Promoções',
        'Qualidade do produto',
        'App / Site / Pagamento',
        'Entrega / Pedido digital',
        'Outros / Genéricos',
    ]
})

categorias_validas

,categoria
0,Atendimento / Loja física
1,Preço / Promoções
2,Qualidade do produto
3,App / Site / Pagamento
4,Entrega / Pedido digital
5,Outros / Genéricos


## 15. Conclusão

A clusterização por LDA produziu uma taxonomia inicial com cinco categorias principais e uma categoria de fallback:

1. Atendimento / Loja física;
2. Preço / Promoções;
3. Qualidade do produto;
4. App / Site / Pagamento;
5. Entrega / Pedido digital;
6. Outros / Genéricos.

A próxima etapa é a anotação manual da amostra por dois anotadores independentes. O cálculo do Kappa de Cohen deve ser feito apenas depois dessa anotação.